# TATN 预训练阶段完整复现（留一法，对照论文）
**评估方式**：9折留一法（Paper Section IV.D），每次用8条完整 drive cycle 训练，留1条测试。

**运行前**：Runtime → Change runtime type → GPU

## Step 1：挂载 Google Drive，设置路径

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_RAW_DATA = '/content/drive/MyDrive/Research/mining review/TATN/Panasonic 18650PF Data'
DRIVE_RESULTS  = '/content/drive/MyDrive/Research/mining review/TATN/results/pretrain_loo'
WORK_DIR = '/content/TATN'

print('根目录内容：')
for item in sorted(os.listdir(DRIVE_RAW_DATA)):
    print(' ', item)

## Step 2：拉取代码

In [ ]:
import shutil
os.chdir('/content')
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
!git clone https://github.com/zhangcastle/TATN.git /content/TATN -q
os.chdir(WORK_DIR)
print('工作目录:', os.getcwd())

## Step 3：安装依赖

In [ ]:
!pip install scipy tqdm scikit-learn -q
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 4：探索原始数据结构

In [ ]:
def find_drive_cycles_dir(base, temp_folder):
    for sub in ['Drive cycles', 'Drive Cycles', 'drive cycles', '']:
        p = os.path.join(base, temp_folder, sub) if sub else os.path.join(base, temp_folder)
        if os.path.exists(p) and any(f.endswith('.mat') for f in os.listdir(p)):
            return p
    return None

temp_raw = {
    '25':  find_drive_cycles_dir(os.path.join(DRIVE_RAW_DATA, 'Panasonic 18650PF Data'), '25degC'),
    '10':  find_drive_cycles_dir(DRIVE_RAW_DATA, '10degC'),
    '0':   find_drive_cycles_dir(DRIVE_RAW_DATA, '0degC'),
    'n10': find_drive_cycles_dir(DRIVE_RAW_DATA, '-10degC'),
    'n20': find_drive_cycles_dir(DRIVE_RAW_DATA, '-20degC'),
}

for temp, path in temp_raw.items():
    print(f'\n=== {temp} ===')
    if path:
        for f in sorted(os.listdir(path)):
            if f.endswith('.mat'): print(' ', f)
    else:
        print('  [未找到]')

## Step 5：数据处理（保存完整 drive cycle，不做 train/test 时间切分）

In [ ]:
import scipy.io as sio
import numpy as np

CYCLE_KW = {
    'Cycle_1': ['Cycle_1','Cycle1'], 'Cycle_2': ['Cycle_2','Cycle2'],
    'Cycle_3': ['Cycle_3','Cycle3'], 'Cycle_4': ['Cycle_4','Cycle4'],
    'NN':      ['_NN_','_NN.'],      'US06':    ['US06'],
    'HWFET':   ['HWFET','HWFTa','HWFTb'],
    'LA92':    ['LA92'],             'UDDS':    ['UDDS'],
}

def match_cycle(fname):
    matched = [c for c, kws in CYCLE_KW.items() if any(kw in fname for kw in kws)]
    return matched[0] if len(matched) == 1 else (f'SKIP' if len(matched) > 1 else None)

def read_mat_file(fpath):
    data = sio.loadmat(fpath)
    if 'meas' in data:
        m = data['meas']
        return (m['Time'][0][0].flatten(), m['Current'][0][0].flatten(),
                m['Voltage'][0][0].flatten(), m['Battery_Temp_degC'][0][0].flatten(),
                m['Ah'][0][0].flatten())
    return (data['time'].flatten(), data['current'].flatten(),
            data['voltage'].flatten(), data['temp'].flatten(), data['ah'].flatten())

def process_temp(raw_path, temp_label, out_base, interval=10):
    """保存完整 drive cycle（不做 train/test 时间切分）"""
    out_path = os.path.join(out_base, temp_label)
    os.makedirs(out_path, exist_ok=True)
    mat_files = sorted([f for f in os.listdir(raw_path) if f.endswith('.mat')])
    all_data = {}
    for fname in mat_files:
        cycle = match_cycle(fname)
        if cycle is None or cycle == 'SKIP':
            if cycle == 'SKIP': print(f'  [跳过合并] {fname}')
            else: print(f'  [未识别] {fname}')
            continue
        t, cur, vol, tmp, ah = read_mat_file(os.path.join(raw_path, fname))
        t2=t[::interval]; cur=cur[::interval]; vol=vol[::interval]
        tmp=tmp[::interval]; ah=ah[::interval]
        idx = next((i for i in range(len(t2)-1) if t2[i+1]-t2[i] < 2), 0)
        seg = (cur[idx:], vol[idx:], tmp[idx:], ah[idx:])
        if cycle in all_data:
            all_data[cycle] = tuple(np.concatenate([all_data[cycle][i], seg[i]]) for i in range(4))
            print(f'  [{temp_label}] 合并: {fname} -> {cycle}  total={len(all_data[cycle][0])}')
        else:
            all_data[cycle] = seg
            print(f'  [{temp_label}] 读取: {fname} -> {cycle}  n={len(seg[0])}')
    if not all_data: print(f'[{temp_label}] 无数据'); return
    # 全局 min/max（按温度）
    def grange(i): v=np.concatenate([d[i] for d in all_data.values()]); return v.min(),v.max()
    ranges = [grange(i) for i in range(4)]
    norm = lambda x,r: (x-r[0])/(r[1]-r[0])
    for cycle,(cur,vol,tmp,ah) in all_data.items():
        cn,vn,tn,an = norm(cur,ranges[0]),norm(vol,ranges[1]),norm(tmp,ranges[2]),norm(ah,ranges[3])
        # 保存完整文件（无切分）
        sio.savemat(os.path.join(out_path, f'{temp_label}{cycle}.mat'),
            {'current':cn.reshape(-1,1),'voltage':vn.reshape(-1,1),
             'temp':tn.reshape(-1,1),'ah':an.reshape(-1,1)})
        print(f'  saved {cycle}.mat  n={len(an)}')
    print(f'[{temp_label}] done\n')

OUTPUT_BASE = os.path.join(WORK_DIR, 'normalized_data', 'Pan')
for temp_label, raw_path in temp_raw.items():
    print(f'========== {temp_label}°C ==========')
    if raw_path: process_temp(raw_path, temp_label, OUTPUT_BASE)
    else: print('路径未找到，跳过\n')
print('全部处理完成')

## Step 6：留一法预训练（5 个温度 × 9 折 = 45 次训练）

每折完成后立刻保存模型到 Drive。预计总耗时 1-2 小时（L4 GPU）。

In [ ]:
import sys, time, glob
import torch.nn as nn, torch.optim as optim
sys.path.insert(0, WORK_DIR)
import importlib
import mydata, models
from pretrain import pretrain_loo
importlib.reload(mydata)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
EPOCHS, BATCH_SIZE, LR, EVAL_INTERVAL = 2000, 64, 0.001, 200
TEMPS = ['25', '10', '0', 'n10', 'n20']
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Device: {DEVICE}  Epochs: {EPOCHS}  Folds: 9 per temp')

In [ ]:
importlib.reload(mydata)
importlib.reload(models)
all_results = {}

for temp in TEMPS:
    print(f'\n{"#"*60}\n# 预训练温度: {temp}°C\n{"#"*60}')
    if not os.path.exists(os.path.join(OUTPUT_BASE, temp)):
        print('数据不存在，跳过'); continue
    t0 = time.time()
    result = pretrain_loo(
        rundir=f'pretrain_loo_{temp}',
        source_temp=temp,
        source_data_path=OUTPUT_BASE + '/',
        all_set=mydata.Pan_all_set,
        lr=LR, batch_size=BATCH_SIZE, epochs=EPOCHS,
        eval_interval=EVAL_INTERVAL, seed=100,
        device_type=DEVICE, ifsave=True,
    )
    elapsed = (time.time()-t0)/60
    all_results[temp] = result
    # 保存所有 fold 的模型到 Drive
    fold_pts = glob.glob(os.path.join(WORK_DIR, f'run/pretrain_loo_{temp}_fold*/saved_model/best.pt'))
    for pt in fold_pts:
        fold_name = pt.split('/')[-3]  # e.g. pretrain_loo_25_fold1_Cycle_1
        dest = os.path.join(DRIVE_RESULTS, f'{fold_name}.pt')
        shutil.copy(pt, dest)
    print(f'[{temp}°C] {len(fold_pts)} 个模型已保存到 Drive，耗时 {elapsed:.1f} min')

print('\n========== 全部完成 ==========')
print('温度  |  avg MAE  |  avg RMSE  |  论文参考')
print('-'*50)
for temp, res in all_results.items():
    if res:
        print(f"{temp:>4}°C | {res['avg_mae']*100:>7.3f}%  | {res['avg_rmse']*100:>8.3f}%   | MAE~1.09% RMSE~1.44%")